# CRUD Operations in ChromaDB: Lifecycle Management & Vector Data Engineering

**Environment:** Google Colab / Local Python 3.9+  
**Domain:** Artificial Intelligence Research & Technical Paper Archive

---

### What You Will Learn
1. **The Vector CRUD Lifecycle:** Understanding the computational mechanics of modifying HNSW graph indices.
2. **CREATE (Ingestion):** Batch ingestion, deterministic content-hashing (SHA-256), and handling duplicate ID collisions.
3. **READ (Retrieval Patterns):** Direct key lookups, metadata filtering, field projections (`include`), pagination (`offset`/`limit`), and multi-query batch search.
4. **UPDATE vs. UPSERT:** When to mutate documents vs. using atomic idempotent upserts in ETL pipelines.
5. **DELETE (Index Pruning):** Targeted ID deletion, conditional metadata purge, and dropping collections.
6. **Production Repository Pattern:** Building a clean, typed `SafePaperRepository` class with error handling and deduplication.
7. **Hands-On Exercises:** Implementing a soft-delete mechanism and automated metadata schema migration.

In [ ]:
# Install ChromaDB and utility libraries
!pip install -q chromadb pandas numpy

## 1. The Vector CRUD Lifecycle

In traditional databases, updating a row updates a B-Tree page. In a vector database, adding, modifying, or deleting a document involves:
- Computing dense embeddings through neural model inference.
- Inserting or rewiring nodes in the multi-layer HNSW (Hierarchical Navigable Small World) proximity graph.
- Managing transactional metadata in an underlying relational or key-value store (SQLite).

| Operation | ChromaDB Method | What Happens Under the Hood |
|---|---|---|
| **Create** | `collection.add()` | Computes embeddings, inserts vectors into HNSW graph, stores metadata. Fails on duplicate ID. |
| **Read** | `collection.get()` / `collection.query()` | `get()` performs exact/metadata filter; `query()` computes nearest neighbors in vector space. |
| **Update** | `collection.update()` | Replaces existing document/metadata. Fails if ID does not exist. |
| **Upsert** | `collection.upsert()` | Idempotent: Updates if ID exists, inserts if ID is new. Golden standard for ETL. |
| **Delete** | `collection.delete()` | Marks vector nodes as deleted, prunes HNSW graph, removes metadata records. |

In [ ]:
import chromadb

# Initialize test client
client = chromadb.Client()

# Helper to reset test collection
def reset_collection(col_name="ai_research_papers"):
    try:
        client.delete_collection(name=col_name)
    except Exception:
        pass
    return client.create_collection(name=col_name, metadata={"hnsw:space": "cosine"})

collection = reset_collection()
print(f"Initialized clean collection '{collection.name}'")

## 2. CREATE: Batch Ingestion & ID Strategies

Every document in ChromaDB must possess a unique `id`.
Two common production ID strategies:
1. **Semantic IDs:** (e.g., `arxiv-2305-18290`)
2. **Deterministic Content Hash:** `hashlib.sha256(text.encode()).hexdigest()[:16]` (guarantees deduplication)

In [ ]:
import hashlib

# Curated dataset of landmark AI research papers
research_papers = [
    {
        "title": "Attention Is All You Need",
        "abstract": "We propose the Transformer architecture based entirely on self-attention mechanisms, dispensing with recurrence and convolutions.",
        "year": 2017,
        "citations": 115000,
        "category": "Architecture",
        "peer_reviewed": True
    },
    {
        "title": "LoRA: Low-Rank Adaptation of Large Language Models",
        "abstract": "LoRA freezes pre-trained model weights and injects trainable rank decomposition matrices into each layer, drastically reducing parameter storage.",
        "year": 2021,
        "citations": 18500,
        "category": "FineTuning",
        "peer_reviewed": True
    },
    {
        "title": "Direct Preference Optimization: Your Language Model is Secretly a Reward Model",
        "abstract": "DPO derives a closed-form solution to optimize policy models directly from human preferences without training a separate reward model or RL.",
        "year": 2023,
        "citations": 4200,
        "category": "Alignment",
        "peer_reviewed": True
    },
    {
        "title": "QLoRA: Efficient Finetuning of Quantized LLMs",
        "abstract": "QLoRA backpropagates gradients through a frozen 4-bit quantized pretrained language model into Low Rank Adapters using NF4 quantization.",
        "year": 2023,
        "citations": 7600,
        "category": "Quantization",
        "peer_reviewed": True
    },
    {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "abstract": "We build RAG models where the parametric memory is a pre-trained seq2seq model and non-parametric memory is a dense vector index of Wikipedia.",
        "year": 2020,
        "citations": 8900,
        "category": "RAG",
        "peer_reviewed": True
    }
]

# Generate deterministic IDs based on title
doc_ids = [f"paper-{hashlib.md5(p['title'].encode()).hexdigest()[:8]}" for p in research_papers]
doc_texts = [f"{p['title']}: {p['abstract']}" for p in research_papers]
doc_metas = [{k: v for k, v in p.items() if k != "abstract"} for p in research_papers]

# Ingest into collection
collection.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metas
)

print(f"Successfully added {collection.count()} papers.")
print(f"Generated IDs: {doc_ids}")

### 2.1 Handling Duplicate ID Collisions with `add()`
Attempting to `add()` an ID that already exists triggers an error or exception.

In [ ]:
try:
    # Attempting to add an existing ID again
    collection.add(
        ids=[doc_ids[0]],
        documents=["Duplicate paper attempt"],
        metadatas=[{"year": 2024}]
    )
    print("Add succeeded (unexpected).")
except Exception as e:
    print(f"Caught expected error on duplicate ID:\n  {type(e).__name__}: {e}")

## 3. READ: Multimodal Retrieval Patterns

ChromaDB provides rich options for inspecting, querying, and paginating records.

### 3.1 Inspection with `peek()` and Count

In [ ]:
print(f"Total Papers in Index: {collection.count()}\n")

# Preview first 2 records
preview = collection.peek(limit=2)
for pid, pdoc in zip(preview["ids"], preview["documents"]):
    print(f"[{pid}] -> {pdoc[:75]}...")

### 3.2 Exact Primary-Key Retrieval with `get()`
`get()` performs lightning-fast primary key lookups without running similarity calculations.

In [ ]:
# Retrieve specific papers by their IDs
target_ids = [doc_ids[0], doc_ids[1]]
records = collection.get(ids=target_ids)

print("Exact ID Lookup Results:")
for pid, pdoc, pmeta in zip(records["ids"], records["documents"], records["metadatas"]):
    print(f"\nID: {pid}")
    print(f"  Title    : {pmeta['title']}")
    print(f"  Category : {pmeta['category']} ({pmeta['year']})")
    print(f"  Citations: {pmeta['citations']}")

### 3.3 Non-Vector Metadata Retrieval
You can also use `get()` with `where` clauses to find documents purely by attributes.

In [ ]:
# Get all papers published in 2023 with citations > 5000
res_filtered = collection.get(
    where={
        "$and": [
            {"year": {"$eq": 2023}},
            {"citations": {"$gt": 5000}}
        ]
    }
)

print(f"Matched {len(res_filtered['ids'])} papers:")
for pid, pmeta in zip(res_filtered["ids"], res_filtered["metadatas"]):
    print(f"  - {pmeta['title']} ({pmeta['citations']} citations)")

### 3.4 Pagination with `limit` and `offset`
When dealing with large collections, pagination avoids fetching millions of records into memory at once.

In [ ]:
# Page 1: 2 records, offset 0
page_1 = collection.get(limit=2, offset=0)
print("Page 1 IDs:", page_1["ids"])

# Page 2: 2 records, offset 2
page_2 = collection.get(limit=2, offset=2)
print("Page 2 IDs:", page_2["ids"])

### 3.5 Semantic Search & Multi-Query Batch Processing
We can issue multiple search queries simultaneously in a single vectorized batch call!

In [ ]:
queries = [
    "Efficient model fine-tuning with low memory rank decomposition",
    "Connecting external Wikipedia knowledge base to neural generators"
]

batch_results = collection.query(
    query_texts=queries,
    n_results=2
)

for q_idx, query in enumerate(queries):
    print(f"\nQuery: '{query}'")
    matched_ids = batch_results["ids"][q_idx]
    matched_docs = batch_results["documents"][q_idx]
    matched_dists = batch_results["distances"][q_idx]
    
    for rank, (mid, mdoc, mdist) in enumerate(zip(matched_ids, matched_docs, matched_dists), start=1):
        print(f"  Rank #{rank} [{mid}] Dist: {mdist:.4f} -> {mdoc[:70]}...")

## 4. UPDATE vs. UPSERT Operations

- `update()`: Mutates an existing document or its metadata. **Fails if the ID is missing**.
- `upsert()`: Inserts if new, replaces if existing. **Safe and idempotent**.

In [ ]:
# 1. Targeted UPDATE: Updating citation count for LoRA
lora_id = doc_ids[1]

collection.update(
    ids=[lora_id],
    metadatas=[{
        "title": "LoRA: Low-Rank Adaptation of Large Language Models",
        "year": 2021,
        "citations": 25000,   # Updated from 18,500 to 25,000!
        "category": "FineTuning",
        "peer_reviewed": True
    }]
)

updated_record = collection.get(ids=[lora_id])
print(f"Updated citations for LoRA: {updated_record['metadatas'][0]['citations']}")

### 4.1 Demonstrating `upsert()`
Let us run an `upsert()` call that simultaneously updates an existing paper (DPO) and adds a brand-new paper (FlashAttention).

In [ ]:
new_paper_id = "paper-flashattn"
dpo_id = doc_ids[2]

upsert_ids = [dpo_id, new_paper_id]
upsert_docs = [
    "DPO: Direct Preference Optimization updated with empirical benchmarks on Llama 3.",
    "FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness using GPU SRAM tiling."
]
upsert_metas = [
    {"title": "DPO (Updated)", "year": 2024, "citations": 6500, "category": "Alignment", "peer_reviewed": True},
    {"title": "FlashAttention", "year": 2022, "citations": 12000, "category": "Architecture", "peer_reviewed": True}
]

print(f"Count before upsert: {collection.count()}")

collection.upsert(
    ids=upsert_ids,
    documents=upsert_docs,
    metadatas=upsert_metas
)

print(f"Count after upsert : {collection.count()} (1 modified, 1 inserted)")
print("Verified FlashAttention added:", collection.get(ids=[new_paper_id])["documents"][0])

## 5. DELETE: Index Pruning & Collection Cleanup

ChromaDB supports two forms of deletion:
1. **By ID List:** `collection.delete(ids=[...])`
2. **By Metadata Filter:** `collection.delete(where={...})`

In [ ]:
# 1. Delete by ID
paper_to_remove = doc_ids[3] # QLoRA
print(f"Count before ID delete: {collection.count()}")
collection.delete(ids=[paper_to_remove])
print(f"Count after deleting '{paper_to_remove}': {collection.count()}")

# 2. Delete by Metadata Filter (e.g. remove all Alignment papers)
print("\nDeleting all papers with category == 'Alignment'...")
collection.delete(where={"category": "Alignment"})
print(f"Count after conditional delete: {collection.count()}")
print("Remaining IDs:", collection.get()["ids"])

## 6. Production Architecture: `SafePaperRepository` Class

In production software, direct database client calls should be encapsulated in a strongly-typed repository pattern.

In [ ]:
from typing import List, Dict, Any, Optional

class SafePaperRepository:
    """Production wrapper around ChromaDB providing validated CRUD operations."""
    
    def __init__(self, collection):
        self.col = collection

    def ingest(self, paper_id: str, text: str, metadata: Dict[str, Any]) -> str:
        """Idempotent upsert ensuring no duplicate key crashes."""
        self.col.upsert(ids=[paper_id], documents=[text], metadatas=[metadata])
        return paper_id

    def fetch(self, paper_id: str) -> Optional[Dict[str, Any]]:
        res = self.col.get(ids=[paper_id])
        if not res["ids"]:
            return None
        return {
            "id": res["ids"][0],
            "document": res["documents"][0],
            "metadata": res["metadatas"][0]
        }

    def search(self, query: str, top_k: int = 3, category: Optional[str] = None):
        where_clause = {"category": category} if category else None
        return self.col.query(query_texts=[query], n_results=top_k, where=where_clause)

    def count(self) -> int:
        return self.col.count()

# Test the repository
repo = SafePaperRepository(collection)
repo.ingest("repo-test-1", "Speculative Decoding for 2x faster LLM inference", {"category": "Inference", "year": 2023})
print("Repository Record Count:", repo.count())
print("Fetched Record:", repo.fetch("repo-test-1")["document"])

## 7. Practice Exercises (Solved)

### Exercise 1: Metadata Schema Migration
Write a script that iterates over all papers in the collection and adds a new metadata field `schema_version = 2` without modifying existing fields.

In [ ]:
# Fetch all existing documents
all_docs = collection.get()

updated_metas = []
for m in all_docs["metadatas"]:
    new_m = dict(m)
    new_m["schema_version"] = 2
    updated_metas.append(new_m)

# Update metadata in batch
collection.update(
    ids=all_docs["ids"],
    metadatas=updated_metas
)

print("Migration Complete. Sample updated metadata:")
print(collection.get(ids=[all_docs["ids"][0]])["metadatas"][0])

### Exercise 2: Implementing Soft-Deletes
Instead of permanently deleting vectors, enterprise systems often set `{"is_deleted": True}`. Implement a soft-delete function.

In [ ]:
def soft_delete(col, doc_id):
    curr = col.get(ids=[doc_id])
    if curr["ids"]:
        meta = curr["metadatas"][0]
        meta["is_deleted"] = True
        col.update(ids=[doc_id], metadatas=[meta])
        print(f"Soft-deleted document '{doc_id}'")

soft_delete(collection, "paper-flashattn")

# Query only active (non-deleted) papers
active_papers = collection.get(where={"is_deleted": {"$ne": True}})
print("Active Paper IDs:", active_papers["ids"])

## Key Takeaways

1. **Upsert is Best for Pipelines:** Prefer `upsert()` over `add()` in continuous ingestion pipelines to prevent crashes on re-indexing.
2. **Selective Includes:** Use `include=["documents", "metadatas"]` on `get()` and `query()` calls to avoid transferring large embedding arrays over IPC when only text is needed.
3. **Soft-Deletes for Auditability:** Use metadata flags (`is_deleted=True`) for reversible deletions in compliance-heavy environments.
4. **Batch Operations:** Always pass lists of documents and IDs rather than making single-item calls to maximize embedding and vector indexing throughput.